# T2 — `T.Oc140k.roles.e8 → Oc`: the same run at four times the data

T1 answered whether removing `?` rows from the training data beats forcing a number at
decode time, and it did, by a wide margin: temperature within +/-10 C rose from 26.1% to
51.6% (top-5), and abstention on top-1 fell from 86.5% to 0.9%. The number was not merely
unspoken -- the model knew it less well when three quarters of its training targets said `?`.

But T1 paid for that with 3.6x less data, and every other field halved: reagents 21.5 ->
9.9%, solvent 33.8 -> 18.9%, catalyst 28.8 -> 14.7%. That trade was the point of the run, not
a flaw in it, and this run removes it. A fresh 900,000-reaction ORD pool yields 139,922
training rows that all carry a temperature -- **more rows than B3 had in total**, with none
of them teaching the model to abstain.

So the comparison is clean in both directions: against T1 it isolates data volume, against B3
it isolates what the `?` rows were doing.

**Leakage check, done before upload, not assumed.** The enlarged pool was drawn from ORD
afresh and excludes only Model 1's ORD eval targets, so it recovered 1,402 training rows and
94 validation rows whose product sits in the conditions test set. All 1,496 are removed here;
the 5,687-record test set is untouched and identical to the one B2, B3 and T1 were scored on.

**Eight epochs, not six.** B3 and T1 both stopped at the very edge of their epoch budget with
`eval_loss` still falling (B3 best at 5.99 of 5.99, T1 at 5.86 of 5.86), so six was the
binding constraint rather than convergence.

**Data:** `kuzmenkoiryna/retro-planner-ord-conditions-temp141k`.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob, json

train_file = next(glob.iglob("/kaggle/input/**/conditions_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/conditions_val.jsonl", recursive=True))
test_file = next(glob.iglob("/kaggle/input/**/conditions_test_clean.jsonl", recursive=True))
for path in (train_file, val_file, test_file):
    print(path, sum(1 for _ in open(path)), "rows")

# The roles split must have travelled with the data, not been re-derived here.
sample = json.loads(open(test_file).readline())
assert "reagents" in sample and "full_reactants_smiles" in sample, "dataset is the pre-roles one"
print("\ninput   :", sample["reactants_smiles"])
print("reagents:", sample["reagents"])
print("as ORD wrote it:", sample["full_reactants_smiles"])

base_model = "t5-small"   # set from B2: wins solvent and catalyst at p<=0.0001
learning_rate = 5e-4
condition_fields = "reagents,solvent,catalyst,temperature_celsius"
output_dir = "/kaggle/working/model2_conditions_temp141k"
time_budget_minutes = 330

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

# Batch 32 x 1, the script's default: t5-small fits without the accumulation split that
# a 220M model needs on a T4. Effective batch stays 32, matching B2.
!torchrun --nproc_per_node=2 scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model2_temp141k \
    --target-format compact \
    --condition-fields "{condition_fields}" \
    --max-source-length 256 \
    --max-target-length 256 \
    --learning-rate {learning_rate} \
    --num-train-epochs 8 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
!grep -E "new character token|Train examples|condition field" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"
marker = json.load(open(f"{output_dir}/final/conditions_format.json"))
print("format marker:", marker)
assert marker["fields"] == condition_fields.split(","), "marker disagrees with the requested fields"

In [ ]:
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 12)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
best_epoch = min(points, key=lambda p: p[1])[0]
print(f"  best {state.get('best_metric')} at epoch {best_epoch:.2f} of {points[-1][0]:.2f} reached")

In [ ]:
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{output_dir}/final" \
    --test-file "{test_file}" \
    --num-beams 10 --batch-size 32 --device cuda \
    --max-source-length 256 --max-target-length 256 \
    --force-numeric \
    --output "/kaggle/working/T2_conditions_temp141k_clean_topk.json"

In [ ]:
import json
data = json.load(open("/kaggle/working/T2_conditions_temp141k_clean_topk.json"))
summary = data["summary"]
print(json.dumps(summary, indent=2))
print("per-record entries kept for a paired test:", len(data["records"]))

# The criterion, applied as written above rather than reinterpreted now.
reagents_ok = (summary.get("reagents_exact_match_top5") or 0) >= 0.30
print(f"\nreagents strict top-5 = {summary.get('reagents_exact_match_top5')} -> "
      f"{'meets' if reagents_ok else 'misses'} the 30% bar set before the run")
for field in ("solvent", "catalyst"):
    print(f"  {field} strict top-5 = {summary.get(field + '_exact_match_top5')} (compare with B2)")